# 07 — ydata-profiling do conjunto final

Profile das **6 features finais** (`03_feature_selection` → `data/processed/features_selecionadas.json`)
no conjunto final de genomas (treino + validação + teste = 2.226), o mesmo que entra no clustering,
na classificação e na regressão.

Comparação com a v1: `reports/features_antes_estruturacao.html` e
`reports/features_depois_estruturacao.html` (`old/10_v1_ydata_estruturacao.ipynb`).

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def encontrar_raiz_repo(marcadores=(".git", "data")):
    for pasta in [Path.cwd(), *Path.cwd().parents]:
        if all((pasta / m).exists() for m in marcadores):
            return pasta
    raise FileNotFoundError(f"Nao achei {marcadores} subindo a partir de {Path.cwd()}")


RAIZ = encontrar_raiz_repo()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

pd.set_option("display.width", 200)

In [2]:
from ydata_profiling import ProfileReport

/tmp/ipykernel_10405/44057814.py:1: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


## 1. Dados

Os 3 splits de `df_estruturada/df_*_risco.csv`, empilhados com uma coluna `conjunto`. `Species` vem
junto só para conferência: não é feature.

In [3]:
pasta = RAIZ / "data/processed/df_estruturada"
df_final = pd.concat({
    "treino": pd.read_csv(pasta / "df_train_risco.csv", index_col=0),
    "validação": pd.read_csv(pasta / "df_valid_risco.csv", index_col=0),
    "teste": pd.read_csv(pasta / "df_test_risco.csv", index_col=0),
}, names=["conjunto", "sample"]).reset_index(level="conjunto")

FEATURES = json.loads((RAIZ / "data/processed/features_selecionadas.json").read_text())["features_finais"]

print(f"{len(df_final)} genomas | {len(FEATURES)} features finais")
print("FEATURES:", FEATURES)
df_final["conjunto"].value_counts()

2226 genomas | 6 features finais
FEATURES: ['has_carbapenemase', 'has_mbl', 'has_esbl', 'n_bl_adquiridas', 'has_16s_metilase', 'carbapenemase_plasmidial']


conjunto
treino       1558
validação     445
teste         223
Name: count, dtype: int64

In [4]:
df_final[FEATURES].head()

,has_carbapenemase,has_mbl,has_esbl,n_bl_adquiridas,has_16s_metilase,carbapenemase_plasmidial
sample,,,,,,
GCA_029076165.1,1,0,0,2,0,1
GCF_001420335.1,1,0,0,2,0,0
GCA_002304205.1,0,0,1,1,0,0
GCA_031011085.1,1,0,1,4,0,1
GCA_031630615.1,1,1,1,3,0,0


## 2. Checagens antes do profile

- **Nulos**: devem ser 0. Gene ausente = 0 (`reindex(fill_value=0)` em `build_features_risco`), não NaN.
- **Linhas repetidas**: com 5 binárias e 1 contagem, vários genomas **diferentes** têm o mesmo perfil.
  São isolados reais, então não são removidos (o ydata vai acusar como "duplicates").
- **Prevalência**: nenhuma binária abaixo de 5% (filtro do `03`).

In [5]:
binarias = [c for c in FEATURES if set(df_final[c].unique()) <= {0, 1}]

checagem = pd.DataFrame({
    "tipo": ["binária" if c in binarias else "contagem" for c in FEATURES],
    "nulos": df_final[FEATURES].isna().sum(),
    "frac_nao_zero": (df_final[FEATURES] > 0).mean().round(3),
    "min": df_final[FEATURES].min(),
    "max": df_final[FEATURES].max(),
})
checagem

,tipo,nulos,frac_nao_zero,min,max
has_carbapenemase,binária,0,0.640,0,1
has_mbl,binária,0,0.108,0,1
has_esbl,binária,0,0.484,0,1
n_bl_adquiridas,contagem,0,0.871,0,7
has_16s_metilase,binária,0,0.136,0,1
carbapenemase_plasmidial,binária,0,0.476,0,1


In [6]:
n_dup = df_final[FEATURES].duplicated().sum()
n_perfis = len(df_final[FEATURES].drop_duplicates())
print(f"{n_dup} linhas repetidas | {n_perfis} perfis distintos em {len(df_final)} genomas")

# perfis mais frequentes
df_final.groupby(FEATURES).size().sort_values(ascending=False).head(10).rename("n_genomas").reset_index()

2159 linhas repetidas | 67 perfis distintos em 2226 genomas


,has_carbapenemase,has_mbl,has_esbl,n_bl_adquiridas,has_16s_metilase,carbapenemase_plasmidial,n_genomas
0,0,0,0,0,0,0,287
1,1,0,1,4,0,1,212
2,0,0,0,1,0,0,148
3,1,0,1,3,0,1,140
4,1,0,0,2,0,1,124
5,1,0,0,1,0,0,115
6,0,0,1,2,0,0,112
7,0,0,1,3,0,0,110
8,1,0,1,3,1,1,103
9,1,0,0,1,0,1,101


### 2.1 Os 3 conjuntos ficaram parecidos?

Média de cada feature por conjunto (split estratificado por espécie).

In [7]:
df_final.groupby("conjunto")[FEATURES].mean().T.round(3)[["treino", "validação", "teste"]]

conjunto,treino,validação,teste
has_carbapenemase,0.644,0.622,0.641
has_mbl,0.116,0.074,0.121
has_esbl,0.485,0.490,0.466
n_bl_adquiridas,2.167,2.196,2.197
has_16s_metilase,0.137,0.124,0.148
carbapenemase_plasmidial,0.481,0.458,0.471


## 3. Profile

As binárias entram como `int` 0/1; o ydata as trata como categóricas. `n_bl_adquiridas` fica numérica.

In [8]:
profile_final = ProfileReport(
    df_final[FEATURES],
    title="Features finais -- conjunto final (treino + validação + teste)",
    minimal=False,
)
profile_final.to_file(RAIZ / "reports/features_finais.html")

/home/lusa/miniconda3/envs/argos/lib/python3.10/site-packages/ydata_profiling/utils/dataframe.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={"index": "df_index"}, inplace=True)


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

100%|██████████| 6/6 [00:00<00:00, 74235.47it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

### 3.1 Leitura dos alertas do ydata (`reports/features_finais.html`)

| alerta | leitura | decisão |
|---|---|---|
| **55 duplicate rows (2,5%)** | o ydata conta os **perfis** que se repetem: 55 dos 67 perfis distintos aparecem em mais de 1 genoma (2.159 linhas são repetições, seção 2) | mantidas: são isolados diferentes com o mesmo perfil, e remover distorceria a prevalência de cada perfil |
| **has_carbapenemase ↔ carbapenemase_plasmidial** | `carbapenemase_plasmidial` só pode ser 1 se `has_carbapenemase` for 1 (ρ = 0,71) | mantidas: abaixo do corte de 0,8 do `03`; uma diz *se tem*, a outra *se é móvel* |
| **has_esbl ↔ n_bl_adquiridas** | a ESBL entra na contagem de beta-lactamases adquiridas (ρ = 0,73) | mantidas: tipo de mecanismo × carga |
| **has_mbl imbalanced (50,7%)** | MBL em 10,8% dos genomas | mantida: acima do filtro de 5% e é o mecanismo mais grave (escapa de ceftazidima-avibactam), define o C0 |
| **n_bl_adquiridas 12,9% zeros** | genomas sem nenhuma beta-lactamase adquirida | zero real, não dado faltante (grupo basal, C5) |

Sem nulos (0 missing cells). As 5 binárias aparecem como *Categorical* e `n_bl_adquiridas` como *Numeric*.

## 4. Correlação (Spearman) das features finais

Conferência de que nenhum par ficou com |ρ| > 0,8 (filtro de redundância do `03`).

In [9]:
corr = df_final[FEATURES].corr("spearman")
fora_diagonal = corr.where(~np.eye(len(FEATURES), dtype=bool))
print(f"maior |ρ| fora da diagonal: {fora_diagonal.abs().max().max():.3f}")
corr.round(2)

maior |ρ| fora da diagonal: 0.729


,has_carbapenemase,has_mbl,has_esbl,n_bl_adquiridas,has_16s_metilase,carbapenemase_plasmidial
has_carbapenemase,1.00,0.26,0.10,0.50,0.23,0.71
has_mbl,0.26,1.00,0.16,0.32,-0.03,0.23
has_esbl,0.10,0.16,1.00,0.73,0.04,0.32
n_bl_adquiridas,0.50,0.32,0.73,1.00,0.04,0.60
has_16s_metilase,0.23,-0.03,0.04,0.04,1.00,0.07
carbapenemase_plasmidial,0.71,0.23,0.32,0.60,0.07,1.00
